In [1]:
import pathlib
# from typing import List

# import numpy as np
import pandas as pd
from IPython.display import display

# import jax
# import jax.numpy as jnp
import jax.random as jr
# import optax
# import equinox as eqx
# import numpyro.distributions as dist

# import torch

import altair as alt

import trapallada.data
import trapallada.model.signal
import trapallada.util
import trapallada.core

# import ipdb

/home/manu/mambaforge/envs/trap/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# TODO

- normalize the signals so that they are between 0 and 1, and then `tanh` en el codificador

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
input_directory = pathlib.Path.cwd().parent / 'data'
assert input_directory.exists()

In [4]:
# base_filename = 'small'
base_filename = '4_classes_modulated'

signals_file = input_directory / f'{base_filename}_signals.pt'
assert signals_file.exists()

metadata_file = input_directory / f'{base_filename}_labels.parquet'
assert metadata_file.exists

In [5]:
# ds = trapallada.data.SignalsDataset(signals_file, metadata_file)
ds = trapallada.data.SignalsDataset(signals_file, metadata_file, normalize=True)
len(ds)

12983

In [6]:
parameters = {
    'training': {
        'splitting': {
            'test_size':  0.2,
            'random_state': 42
        },
        'batch_size': 32,
        # 'learning_rate': 1e-3,
        'learning_rate': 3e-5,
        # 'learning_rate': 1e-6,
        'dl_rng_key': 43,
    },
}

In [7]:
train_dl, validation_dl = trapallada.data.dataloaders_from_dataset(ds, parameters['training']['splitting'], parameters['training']['batch_size'], parameters['training']['dl_rng_key'])

In [8]:
batch = next(iter(train_dl))

In [9]:
signals_len = ds[0][1].shape[-1]
signals_len

10000

In [10]:
tuple(ds[0][1].shape)

(1, 10000)

In [11]:
parameters['encoder'] = {
    'input_size': ds[0][1].shape,
}

In [12]:
parameters['model'] = {
    # 'dim_z': 4,
    'dim_z': 10,
    # 'dim_z': 20,
    'kernel_sizes': [128, 64, 32],
    'channels': [12, 12, 12],
    'stride': 2,
}

In [13]:
x = batch[1][0].numpy()
encoder = trapallada.model.signal.Encoder(**(parameters['model'] | parameters['encoder']), key=jr.PRNGKey(42))
encoder.debug_call(x)
print(encoder)

conv_output_len=(4937,)
conv_output_len=(2437,)
conv_output_len=(1203,)
after equinox.nn._conv: (12, 4937)
after jax._src.nn.functions: (12, 4937)
after equinox.nn._normalisation: (12, 4937)
after equinox.nn._conv: (12, 2437)
after jax._src.nn.functions: (12, 2437)
after equinox.nn._normalisation: (12, 2437)
after equinox.nn._conv: (12, 1203)
after jax._src.nn.functions: (12, 1203)
after equinox.nn._normalisation: (12, 1203)
after jax._src.numpy.lax_numpy: (14436,)
after equinox.nn._linear: (20,)
after trapallada.model.signal: (20,)
Encoder(
  layers=[
    Conv1d(
      num_spatial_dims=1,
      weight=f32[12,1,128],
      bias=f32[12,1],
      in_channels=1,
      out_channels=12,
      kernel_size=(128,),
      stride=(2,),
      padding=((0, 0),),
      dilation=(1,),
      groups=1,
      use_bias=True
    ),
    <wrapped function relu>,
    LayerNorm(
      shape=(12, 4937),
      eps=1e-05,
      use_weight=True,
      use_bias=True,
      weight=f32[12,4937],
      bias=f32[12,4

In [14]:
encoder.layers_spatial_sizes

[10000, 4937, 2437, 1203, 14436]

In [15]:
z = encoder(x)
z.shape

(20,)

In [16]:
parameters['decoder'] = {'layers_spatial_sizes': encoder.layers_spatial_sizes}

In [17]:
z_mean = jr.normal(jr.PRNGKey(42), (parameters['model']['dim_z'],))
decoder = trapallada.model.signal.Decoder(**(parameters['model'] | parameters['decoder']), key=jr.PRNGKey(42))
decoder.debug_call(z_mean)

after equinox.nn._linear: (14436,)
after jax._src.nn.functions: (14436,)
after trapallada.model.signal: (12, 1203)
after equinox.nn._conv: (12, 2437)
after jax._src.nn.functions: (12, 2437)
after equinox.nn._normalisation: (12, 2437)
after equinox.nn._conv: (12, 4937)
after jax._src.nn.functions: (12, 4937)
after equinox.nn._normalisation: (12, 4937)
after equinox.nn._conv: (1, 10000)


In [18]:
vae = trapallada.model.signal.VAE(encoder, decoder)
x_pred, z_mean, z_sd = vae(x, jr.PRNGKey(42))
x_pred.shape, z_mean.shape, z_sd.shape

((1, 10000), (10,), (10,))

In [19]:
x_pred

Array([[-0.04602344,  0.02622148, -0.04247404, ...,  0.00272043,
        -0.00598365, -0.02058782]], dtype=float32)

In [20]:
parameters['training']['n_epochs'] = 3
parameters['loss'] = {'x_sd': 0.5}

In [21]:
model, results_df = trapallada.core.fit_vae_on_signals(train_dl, validation_dl, parameters)

conv_output_len=(4937,)
conv_output_len=(2437,)
conv_output_len=(1203,)
       training                             validation                \
           loss loglikelihood         kl          loss loglikelihood   
0  21167.439549 -21141.769369  25.670203  18041.262933 -17998.479468   

              
          kl  
0  42.783432  
       training                            validation                         
           loss loglikelihood        kl          loss loglikelihood         kl
1  16835.452794 -16789.411749  46.04106  15796.209663 -15745.656512  50.553124
       training                            validation                         
           loss loglikelihood        kl          loss loglikelihood         kl
2  15418.258624 -15364.941689  53.31696  14915.767816 -14859.326351  56.441424


In [22]:
i = 4
x_valid = next(iter(train_dl))[1].numpy()
x_pred, z_mean, z_sd = model(x_valid[i], jr.PRNGKey(42))

df = pd.DataFrame({'prediction': x_pred[0], 'actual': x_valid[i].ravel()})
n_rows = 2000
alt.Chart(
    df.iloc[:n_rows].reset_index()
).transform_fold(
    df.columns.to_list()
).mark_line().encode(
    x='index:Q', y='value:Q', color='key:N'
).properties(width=1_200, height=400)

alt.Chart(...)

In [23]:
z_mean, z_sd

(Array([ 4.556385 , -2.3994546, -2.4583738,  4.519268 , -8.260303 ,
        -1.1589789, -1.1604321,  0.3365258, -2.2928853, -1.1339872],      dtype=float32),
 Array([0.0265237 , 0.01598905, 0.02173913, 0.01281478, 0.02866968,
        0.01708689, 0.04568901, 0.00775277, 0.02719858, 0.01812593],      dtype=float32))

In [25]:
model.encoder(x_valid[0])

Array([-4.9606113 ,  2.52772   ,  0.4861421 , -0.00740718, -0.11631282,
       -1.7101355 , -0.46153015, -1.6973938 ,  2.403687  , -2.4160392 ,
        0.03400524,  0.02266127,  0.02134818,  0.03738111,  0.03348738,
        0.04584688,  0.02097749,  0.03641305,  0.02209009,  0.03295711],      dtype=float32)

*Untrained* encoder

In [26]:
encoder(x_valid[0])

Array([-0.7117197 ,  1.990339  ,  0.7993523 ,  0.96170557, -0.17030774,
        1.1468799 , -0.66805375,  0.04921728,  0.6763457 , -0.4492945 ,
        0.8475532 ,  0.988918  ,  0.30313388,  1.2485397 ,  0.3529517 ,
        0.7225193 ,  0.8317149 ,  0.863597  ,  0.6062124 ,  0.5473657 ],      dtype=float32)